# Practice Lab: Prompting Fundamentals

Module 5 \u00b7 Lesson 5.1 \u00b7 4 exercises
**Needs:** google-generativeai, pydantic


# Lesson 5.1: Prompting Fundamentals

4 exercises: 2E + 1M + 1H
**Needs:** `pip install google-generativeai pydantic`

All exercises use Gemini API.


In [ ]:
!pip install google-generativeai pydantic -q


In [ ]:
import google.generativeai as genai
import json, os

genai.configure(
    api_key=os.getenv('GOOGLE_API_KEY'))

def ask(prompt, temperature=0.3):
    model = genai.GenerativeModel(
        'gemini-2.0-flash',
        generation_config={
            'temperature': temperature})
    return model.generate_content(prompt).text

# Quick test
print(ask('Say hello in one word.'))


---
## Exercise 1 (Easy): Prompt Clarity Test
Write 3 prompt versions: vague, medium, specific.
Compare word count and usefulness.


In [ ]:
# YOUR CODE
prompts = {
    "Vague": "Give me Python tips.",
    "Medium": "Give me 5 Python tips for beginners.",
    "Specific": (
        "Give me exactly 5 Python tips for a developer "
        "with 1 year of experience who wants to write "
        "cleaner code. For each tip: one sentence "
        "explanation + one code example under 3 lines. "
        "Do NOT include basic syntax tips."
    ),
}

# TODO: run all 3, compare word count
# TODO: rate usefulness 1-5


<details><summary>Solution</summary>


In [ ]:
results = {}
for level, prompt in prompts.items():
    result = ask(prompt)
    words = len(result.split())
    results[level] = {
        "words": words,
        "preview": result[:150],
    }
    print(f"\n{"="*50}")
    print(f"LEVEL: {level} ({words} words)")
    print(f"{"="*50}")
    print(result[:300])

print(f"\n{"Level":<10} {"Words":>6}")
print("-" * 20)
for level, r in results.items():
    print(f"  {level:<8} {r["words"]:>6}")

print("\nSpecific prompt: fewer words, ""MORE useful!")


</details>


---
## Exercise 2 (Easy): Role Gallery
Ask same question to 5 roles, compare outputs.


In [ ]:
# YOUR CODE
question = "What is cloud computing?"

roles = [
    ("Teacher", "You are a high school CS teacher. Explain simply in 3 sentences."),
    ("5-year-old", "You are a 5-year-old child. Explain in your own words."),
    ("CEO", "You are a Fortune 500 CEO. Explain business value in 3 sentences."),
    ("Poet", "You are a poet. Explain through a short poem (4 lines)."),
    ("Commentator", "You are a cricket commentator. Explain using cricket analogies."),
]

# TODO: run each, print output + word count
# TODO: rate accuracy/creativity/usefulness


<details><summary>Solution</summary>


In [ ]:
for name, role in roles:
    prompt = f"{role}\n\n{question}"
    result = ask(prompt, temperature=0.7)
    print(f"\n{"="*45}")
    print(f"ROLE: {name}")
    print(f"{"="*45}")
    print(result.strip())
    print(f"  [{len(result.split())} words]")


</details>


---
## Exercise 3 (Medium): News Headline Classifier
Build JSON classifier, test on 10 headlines.


In [ ]:
# YOUR CODE
headlines = [
    ("India wins T20 World Cup", "sports"),
    ("New AI chip doubles speed", "tech"),
    ("PM announces highway project", "politics"),
    ("Bollywood star launches company", "entertainment"),
    ("RBI holds interest rates", "business"),
    ("Tesla stock surges 15%", "business"),
    ("ISRO launches satellite", "tech"),
    ("Opposition calls for debate", "politics"),
    ("IPL auction sets record", "sports"),
    ("Netflix announces Hindi series", "entertainment"),
]

# TODO: classify each headline via Gemini
# TODO: validate JSON, calculate accuracy


<details><summary>Solution</summary>


In [ ]:
model = genai.GenerativeModel(
    "gemini-2.0-flash",
    system_instruction=(
        "You are a news classifier. Return ONLY "
        "valid JSON. No markdown, no extra text."),
    generation_config={"temperature": 0.0})

correct, valid = 0, 0
for headline, expected in headlines:
    prompt = (
        f'Classify: "{headline}"\n'
        f'Return JSON: {{"category":"...",'
        f'"sentiment":"...",'
        f'"summary":"..."}}\n'
        f'category: politics, sports, tech, '
        f'entertainment, business')
    resp = model.generate_content(
        prompt).text.strip()
    try:
        text = resp.strip('`').replace(
            'json\n', '')
        data = json.loads(text)
        valid += 1
        cat = data.get("category","").lower()
        ok = '\u2705' if cat == expected \
            else f'\u274c (got {cat})'
        if cat == expected: correct += 1
    except:
        ok = '\u274c (bad JSON)'
    print(f'  {ok} {headline[:40]}...')

print(f'\nValid JSON: {valid}/'
      f'{len(headlines)}')
print(f'Accuracy:   {correct}/'
      f'{len(headlines)} '
      f'({correct/len(headlines):.0%})')


</details>


---
## Exercise 4 (Challenge): Prompt Reliability Benchmark
3 prompt variants x 20 runs each. Measure reliability.


In [ ]:
# YOUR CODE
review = ("Amazing battery life but the screen "
          "is dim. 3 out of 5 stars.")

# Variant A: minimal
prompt_a = (
    f'Analyze sentiment: "{review}"\n'
    f'Return JSON.')

# Variant B: detailed with schema
prompt_b = (
    f'Analyze this review sentiment.\n'
    f'Review: "{review}"\n'
    f'Return ONLY valid JSON (no markdown):\n'
    f'{{"sentiment":"positive|negative|mixed",'
    f'"confidence":0.0-1.0,'
    f'"summary":"one sentence"}}')

# Variant C: system-style with rules
prompt_c = (
    f'You are a sentiment analyzer.\n'
    f'Rules: Return ONLY valid JSON.\n'
    f'Schema: {{"sentiment":"positive|negative|mixed",'
    f'"confidence":float 0-1,'
    f'"summary":"under 100 chars"}}\n'
    f'Review: "{review}"')

# TODO: run each 20 times
# TODO: check json/sentiment/confidence/summary
# TODO: print reliability table


<details><summary>Solution</summary>


In [ ]:
def test_prompt(name, prompt, n=20):
    checks = {"json": 0, "sentiment": 0,
              "confidence": 0, "summary": 0}
    for _ in range(n):
        r = ask(prompt)
        try:
            text = r.strip('`').replace(
                'json\n', '')
            d = json.loads(text)
            checks["json"] += 1
            if d.get('sentiment') in [
                'positive','negative','mixed']:
                checks["sentiment"] += 1
            if isinstance(d.get('confidence'),
                          (int, float)):
                checks["confidence"] += 1
            if len(str(
                d.get('summary',''))) > 5:
                checks["summary"] += 1
        except:
            pass
    print(f'\n  {name}:')
    for k, v in checks.items():
        pct = v / n * 100
        e = '\u2705' if pct >= 90 else (
            '\u26a0\ufe0f' if pct >= 70 \
            else '\u274c')
        print(f'    {e} {k}: {pct:.0f}%')

for name, prompt in [
    ("A-Minimal", prompt_a),
    ("B-Detailed", prompt_b),
    ("C-SystemStyle", prompt_c)]:
    test_prompt(name, prompt, n=20)

print('\nExpected: C wins (explicit rules)')


</details>
